#  EDA + Preparación de datos (Bike Sharing)

"
"**Objetivo:** dejar el dataset claro (head, columnas, tipos, nulos) y preparar `X` / `y` sin fuga de información.

"
"Usamos los archivos locales `hour.csv`

In [ ]:
!pip install ucimlrepo
!pip install pandas scikit-learn streamlit joblib matplotlib

In [65]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt


##  Carga de datos (archivos locales)

In [37]:
import pandas as pd

df = pd.read_csv("../data/raw/hour.csv")
df.shape


(17379, 17)

##  Vista rápida: `head`, columnas, tipos, nulos

In [ ]:
print('Shape:', df.shape)
print('\nColumns:\n', df.columns.tolist())
display(df.head(10))

In [ ]:
df.info()

In [ ]:
display(df.describe().T)

In [ ]:
# Nulos por columna
df.isna().sum().sort_values(ascending=False)

##  Diccionario de variables (hour.csv)

**Temporales / calendario**
- `dteday`: fecha
- `hr`: hora del día (0–23)
- `yr`: año (0=2011, 1=2012)
- `mnth`: mes
- `weekday`: día de semana
- `holiday`: festivo (0/1)
- `workingday`: día laboral (0/1)

**Categóricas codificadas**
- `season`: estación (1–4)
- `weathersit`: situación meteorológica (1–4)

**Clima (normalizadas 0–1 en este dataset)**
- `temp`, `atemp`, `hum`, `windspeed`

**Demanda**
- `casual`, `registered`
- `cnt`: total por hora ✅ **target**

⚠️ Importante: `casual` y `registered` NO se usan como features para predecir `cnt` (fuga de información).


## Conversión de fecha + chequeos

In [ ]:
df['dteday'] = pd.to_datetime(df['dteday'])
print('Duplicados:', df.duplicated().sum())
df.head(3)

In [ ]:
import numpy as np

df["hr_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hr_cos"] = np.cos(2 * np.pi * df["hr"] / 24)

df[["hr", "hr_sin", "hr_cos"]].head()

## EDA mínima del target (cnt) y patrón por hora

In [ ]:
plt.figure()
plt.hist(df['cnt'], bins=60)
plt.title('Distribución de demanda por hora (cnt)')
plt.xlabel('cnt')
plt.ylabel('Frecuencia')
plt.show()

df['cnt'].describe()

In [ ]:
hr_mean = df.groupby('hr')['cnt'].mean().sort_index()
plt.figure()
plt.plot(hr_mean.index, hr_mean.values, marker='o')
plt.title('Demanda promedio por hora (hr)')
plt.xlabel('Hora (hr)')
plt.ylabel('cnt promedio')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()


## Preparación de X / y (sin leakage)

In [45]:
TARGET = 'cnt'
LEAK_COLS = ['casual', 'registered']
DROP_ALWAYS = ['instant', 'dteday']

X = df.drop(columns=[TARGET] + LEAK_COLS + DROP_ALWAYS)
y = df[TARGET].copy()

print('X shape:', X.shape, '| y shape:', y.shape)
print('Features:', X.columns.tolist())
display(X.head())

X shape: (17379, 12) | y shape: (17379,)
Features: ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0


In [53]:
TARGET = "cnt"
LEAK_COLS = ["casual", "registered"]

# acá sacamos hr porque ahora lo representamos con hr_sin/hr_cos
DROP_ALWAYS = ["instant", "dteday", "hr"]

X2 = df.drop(columns=[TARGET] + LEAK_COLS + DROP_ALWAYS)
y  = df[TARGET].copy()

print("X2 shape:", X2.shape, "| y shape:", y.shape)
print("Primeras columnas X2:", X2.columns.tolist())
X2.head()

X2 shape: (17379, 13) | y shape: (17379,)
Primeras columnas X2: ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'hr_sin', 'hr_cos']


,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,hr_sin,hr_cos
0,1,0,1,0,6,0,1,0.24,0.2879,0.81,0.0,0.000000,1.000000
1,1,0,1,0,6,0,1,0.22,0.2727,0.80,0.0,0.258819,0.965926
2,1,0,1,0,6,0,1,0.22,0.2727,0.80,0.0,0.500000,0.866025
3,1,0,1,0,6,0,1,0.24,0.2879,0.75,0.0,0.707107,0.707107
4,1,0,1,0,6,0,1,0.24,0.2879,0.75,0.0,0.866025,0.500000


## Pipeline (OneHot para categóricas + GradientBoosting)

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# listas actualizadas (sin hr, con hr_sin/hr_cos)
categorical_cols = ['season','yr','mnth',
                    'holiday','weekday','workingday','weathersit']

numeric_cols = ['temp','atemp','hum','windspeed',
                'hr_sin','hr_cos']

preprocess2 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop"
)

model2 = GradientBoostingRegressor(random_state=42)

pipe2 = Pipeline(steps=[
    ("preprocess", preprocess2),
    ("model", model2)
])

# Split con X2
X2_train, X2_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

## Entrenamiento + evaluación

In [60]:
pipe2.fit(X2_train, y_train)
pred2 = pipe2.predict(X2_test)

mae2 = mean_absolute_error(y_test, pred2)
r2_2 = r2_score(y_test, pred2)

print("MAE circular:", round(mae2, 2))
print("R² circular:", round(r2_2, 4))

MAE circular: 46.49
R² circular: 0.8642


In [63]:
print("Modelo horario (OneHot hr)   -> MAE: 58.00 | R²: 0.7984")
print("Modelo horario (Circular)    -> MAE:", round(mae2,2), "| R²:", round(r2_2,4))

Modelo horario (OneHot hr)   -> MAE: 58.00 | R²: 0.7984
Modelo horario (Circular)    -> MAE: 46.49 | R²: 0.8642


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

hr_mean = df.groupby("hr")["cnt"].mean().sort_index()

plt.figure()
plt.plot(hr_mean.index, hr_mean.values, marker="o")
plt.title("Demanda promedio por hora (patrón cíclico)")
plt.xlabel("Hora (hr)")
plt.ylabel("cnt promedio")
plt.xticks(range(0,24))
plt.grid(True, alpha=0.3)
plt.show()

## Métrica extra: MAPE aproximado

In [68]:
mask = y_test > 0
mape_filtered = (np.abs(y_test[mask] - pred2[mask]) / y_test[mask]).mean() * 100
print("MAPE sin ceros (%):", round(mape_filtered, 2))

MAPE sin ceros (%): 132.18


## Guardar pipeline

In [67]:
import joblib
joblib.dump(pipe2, "../models/gradient_boosting_hour_circular.pkl")

['../models/gradient_boosting_hour_circular.pkl']